# DES Y3 Tomo-4 Shear Covariance Diagnostic

This notebook diagnoses the DES Y3 source-bin 4 shear auto-spectrum covariance for the transfer-product map

`data/des_y3_shear_maps/des_y3_metacal_shear_maps_nside1024.h5`, group `maps/tomo3`.

The immediate target is the `4x4` shear auto spectrum shown in Fig. 4 of the DES Y3 harmonic-space paper (`2203.07128v1.pdf`). The working hypothesis is that the map is close to correct, while the very broad error bars are caused by an inconsistent NaMaster covariance input: a coupled pseudo-spectrum divided by a scalar mask moment was passed to `gaussian_covariance(..., coupled=False)`.

The notebook tests this in four layers:

1. HDF5 map and metadata sanity checks.
2. Re-pixelization from the processed DES shear pickle.
3. NaMaster spectrum and shape-noise convention checks.
4. NaMaster covariance variants, including the current suspect one and corrected alternatives.

In [2]:
from pathlib import Path
import json
import os
import time
import warnings

os.environ.setdefault('MPLCONFIGDIR', '/tmp/act_desi_ksz_mplconfig')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/act_desi_ksz_xdgcache')

import h5py
import healpy as hp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

try:
    import dill
except Exception as exc:
    dill = None
    print('dill is unavailable:', repr(exc))

try:
    import pymaster as nmt
except Exception as exc:
    nmt = None
    print('pymaster/NaMaster is unavailable:', repr(exc))

try:
    from scipy.ndimage import gaussian_filter1d
except Exception:
    gaussian_filter1d = None


def find_package_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if (cand / 'manifest.json').exists() and (cand / 'data' / 'des_y3_shear_maps').exists():
            return cand
        nested = cand / 'DESI' / 'act_desi_ksz_transfer'
        if (nested / 'manifest.json').exists() and (nested / 'data' / 'des_y3_shear_maps').exists():
            return nested
    raise RuntimeError('Could not find act_desi_ksz_transfer package root from current working directory')


PACKAGE_ROOT = find_package_root()
OUTDIR = PACKAGE_ROOT / 'diagnostics' / 'des_y3_shear_tomo4_covariance'
OUTDIR.mkdir(parents=True, exist_ok=True)

MANIFEST = json.loads((PACKAGE_ROOT / 'manifest.json').read_text())
SHEAR_H5 = PACKAGE_ROOT / MANIFEST['products']['des_y3_shear_maps']['nside1024']
PICKLE_PATH = Path('/global/cfs/cdirs/des/data_actxdes/des_data/cat_DES_shearcat_all_dump_nzfix_Feb25.pk')
BASE_INDEXCAT = Path('/global/cfs/cdirs/lsst/www/shivamp/data_actxdes/Y3_cats/DESY3_indexcat.h5')
REFERENCE_PAPER_PDF = Path('/global/cfs/cdirs/lsst/www/shivamp/DESI/2203.07128v1.pdf')
CURRENT_USER_PLOT = Path('/global/cfs/cdirs/lsst/www/shivamp/DESI/des_shear_tomo4x4_fig4_check.png')
PAPER_PANEL_PLOT = Path('/global/cfs/cdirs/lsst/www/shivamp/DESI/shear_4_4_paper_DES.png')

NSIDE = 1024
TOMO_INDEX = 3
TOMO_LABEL = 'tomo4'
GROUP_NAME = f'maps/tomo{TOMO_INDEX}'

# Expensive switches. The notebook implements all diagnostics, but these let you
# make a fast first pass and then turn on the slow cells deliberately.
RUN_MAP_RECONSTRUCTION_FROM_PICKLE = True
RUN_BASE_CATALOG_AUDIT = False
RUN_NAMASTER_SPECTRA = True
RUN_NAMASTER_COVARIANCE = True
RUN_RANDOM_ROTATIONS = True
N_RANDOM_ROTATIONS = 20
RANDOM_SEED = 271828

summary = {
    'package_root': str(PACKAGE_ROOT),
    'output_directory': str(OUTDIR.relative_to(PACKAGE_ROOT)),
    'shear_h5': str(SHEAR_H5.relative_to(PACKAGE_ROOT)),
    'tomo_index_zero_based': TOMO_INDEX,
    'tomo_label_one_based': TOMO_LABEL,
    'nside': NSIDE,
    'created_unix_time': time.time(),
    'switches': {
        'RUN_MAP_RECONSTRUCTION_FROM_PICKLE': RUN_MAP_RECONSTRUCTION_FROM_PICKLE,
        'RUN_BASE_CATALOG_AUDIT': RUN_BASE_CATALOG_AUDIT,
        'RUN_NAMASTER_SPECTRA': RUN_NAMASTER_SPECTRA,
        'RUN_NAMASTER_COVARIANCE': RUN_NAMASTER_COVARIANCE,
        'RUN_RANDOM_ROTATIONS': RUN_RANDOM_ROTATIONS,
        'N_RANDOM_ROTATIONS': N_RANDOM_ROTATIONS,
    },
    'notes': [],
}


def to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, Path):
        return str(obj)
    return obj


def save_json(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(to_builtin(payload), indent=2, sort_keys=True))
    tmp.replace(path)


def finite_stats(values, mask=None):
    arr = np.asarray(values)
    if mask is not None:
        arr = arr[np.asarray(mask, dtype=bool)]
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return {'n': 0, 'min': None, 'max': None, 'mean': None, 'median': None, 'std': None}
    return {
        'n': int(arr.size),
        'min': float(np.min(arr)),
        'max': float(np.max(arr)),
        'mean': float(np.mean(arr)),
        'median': float(np.median(arr)),
        'std': float(np.std(arr)),
    }


def write_placeholder_plot(path, title, message):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.axis('off')
    ax.text(0.5, 0.62, title, ha='center', va='center', fontsize=15, weight='bold')
    ax.text(0.5, 0.42, message, ha='center', va='center', fontsize=11, wrap=True)
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)

print('Package root:', PACKAGE_ROOT)
print('Diagnostics output:', OUTDIR)
print('Shear HDF5:', SHEAR_H5)
print('NaMaster available:', nmt is not None)

Package root: /global/cfs/cdirs/lsst/www/shivamp/DESI/act_desi_ksz_transfer
Diagnostics output: /global/cfs/cdirs/lsst/www/shivamp/DESI/act_desi_ksz_transfer/diagnostics/des_y3_shear_tomo4_covariance
Shear HDF5: /global/cfs/cdirs/lsst/www/shivamp/DESI/act_desi_ksz_transfer/data/des_y3_shear_maps/des_y3_metacal_shear_maps_nside1024.h5
NaMaster available: True


## 1. Validate the Saved HDF5 Shear Product

This cell loads only tomo bin 4 (`maps/tomo3`) from the `nside=1024` product and checks the internal identities that should hold if the transfer map was written correctly.

In [3]:
map_names = [
    'gamma1',
    'gamma2',
    'gamma2_namaster',
    'mask_weight_raw',
    'mask_weight',
    'mask_binary',
    'count',
    'sum_weight_sq',
    'sum_w2_e2_over2',
]

with h5py.File(SHEAR_H5, 'r') as h5:
    root_attrs = dict(h5.attrs)
    tomo_group = h5[GROUP_NAME]
    tomo_attrs = dict(tomo_group.attrs)
    maps = {name: tomo_group[name][:] for name in map_names}
    ell_left = h5['bandpowers/ell_left'][:].astype(int)
    ell_right = h5['bandpowers/ell_right'][:].astype(int)
    pixwin_pol = h5['pixel_window/polarization'][:]

npix = hp.nside2npix(NSIDE)
assert maps['gamma1'].shape == (npix,), maps['gamma1'].shape
assert int(root_attrs['nside']) == NSIDE
assert int(root_attrs['npix']) == npix

observed = maps['mask_weight_raw'] > 0
outside = ~observed
mean_weight = float(tomo_attrs['mean_weight_per_observed_pixel'])
raw_noise = float(tomo_attrs['shape_noise_pseudo_cl_raw_weight_mask'])
norm_noise = float(tomo_attrs['shape_noise_pseudo_cl_normalized_weight_mask'])
binary_noise = float(tomo_attrs['shape_noise_pseudo_cl_binary_mask'])

sanity = {
    'nside': NSIDE,
    'npix': int(npix),
    'n_valid_sources_attr': int(tomo_attrs['n_valid_sources']),
    'count_sum': int(np.sum(maps['count'], dtype=np.int64)),
    'n_observed_pixels_attr': int(tomo_attrs['n_observed_pixels']),
    'n_observed_pixels_from_mask': int(np.count_nonzero(observed)),
    'area_observed_deg2_binary': float(tomo_attrs['area_observed_deg2_binary']),
    'mean_count_per_observed_pixel': float(tomo_attrs['mean_count_per_observed_pixel']),
    'mean_weight_per_observed_pixel': mean_weight,
    'n_eff_per_arcmin2_binary_area': float(tomo_attrs['n_eff_per_arcmin2_binary_area']),
    'weighted_mean_gamma1_catalog': float(tomo_attrs['weighted_mean_gamma1_catalog']),
    'weighted_mean_gamma2_catalog': float(tomo_attrs['weighted_mean_gamma2_catalog']),
    'max_abs_gamma2_plus_gamma2_namaster': float(np.max(np.abs(maps['gamma2'] + maps['gamma2_namaster']))),
    'max_abs_gamma1_outside_mask': float(np.max(np.abs(maps['gamma1'][outside]))) if np.any(outside) else 0.0,
    'max_abs_gamma2_namaster_outside_mask': float(np.max(np.abs(maps['gamma2_namaster'][outside]))) if np.any(outside) else 0.0,
    'max_abs_mask_weight_raw_scaled_minus_mask_weight': float(
        np.max(np.abs(maps['mask_weight_raw'][observed] / mean_weight - maps['mask_weight'][observed]))
    ),
    'shape_noise_raw': raw_noise,
    'shape_noise_normalized': norm_noise,
    'shape_noise_binary': binary_noise,
    'shape_noise_raw_div_mean_weight_sq': raw_noise / mean_weight**2,
    'shape_noise_scaling_absdiff_raw_to_norm': abs(raw_noise / mean_weight**2 - norm_noise),
    'mask_weight_raw_stats_observed': finite_stats(maps['mask_weight_raw'], observed),
    'mask_weight_stats_observed': finite_stats(maps['mask_weight'], observed),
    'count_stats_observed': finite_stats(maps['count'], observed),
    'gamma1_stats_observed': finite_stats(maps['gamma1'], observed),
    'gamma2_namaster_stats_observed': finite_stats(maps['gamma2_namaster'], observed),
}

sanity['pass_count_matches_attr'] = sanity['count_sum'] == sanity['n_valid_sources_attr']
sanity['pass_observed_pixel_count_matches_attr'] = sanity['n_observed_pixels_from_mask'] == sanity['n_observed_pixels_attr']
sanity['pass_gamma2_sign_flip'] = sanity['max_abs_gamma2_plus_gamma2_namaster'] == 0.0
sanity['pass_shear_zero_outside_mask'] = (
    sanity['max_abs_gamma1_outside_mask'] == 0.0 and sanity['max_abs_gamma2_namaster_outside_mask'] == 0.0
)
sanity['pass_raw_normalized_noise_scaling'] = sanity['shape_noise_scaling_absdiff_raw_to_norm'] < 1e-18

summary['map_sanity'] = sanity
save_json(OUTDIR / 'summary.json', summary)

print(json.dumps(to_builtin(sanity), indent=2, sort_keys=True)[:5000])

{
  "area_observed_deg2_binary": 4731.367114301079,
  "count_stats_observed": {
    "max": 93.0,
    "mean": 17.507000638878456,
    "median": 17.0,
    "min": 1.0,
    "n": 1443154,
    "std": 8.396526544713906
  },
  "count_sum": 25265298,
  "gamma1_stats_observed": {
    "max": 1.1678191423416138,
    "mean": -6.360547558870167e-05,
    "median": -6.877984560560435e-05,
    "min": -1.1415821313858032,
    "n": 1443154,
    "std": 0.09847082197666168
  },
  "gamma2_namaster_stats_observed": {
    "max": 1.2445420026779175,
    "mean": -4.9380330892745405e-05,
    "median": 1.9671133486554027e-05,
    "min": -1.1630170345306396,
    "n": 1443154,
    "std": 0.09833166003227234
  },
  "mask_weight_raw_stats_observed": {
    "max": 2551.147216796875,
    "mean": 358.82769775390625,
    "median": 336.62774658203125,
    "min": 10.300535202026367,
    "n": 1443154,
    "std": 188.0050811767578
  },
  "mask_weight_stats_observed": {
    "max": 7.109669208526611,
    "mean": 0.9999997615814

In [4]:
# Quicklook maps. These are for visual inspection only; use the HDF5 arrays for measurements.
quicklook_path = OUTDIR / 'tomo4_shear_maps_quicklook.png'

shown_mask = np.full(npix, hp.UNSEEN, dtype=np.float32)
shown_mask[observed] = maps['mask_weight'][observed].astype(np.float32)

shown_log_count = np.full(npix, hp.UNSEEN, dtype=np.float32)
shown_log_count[observed] = np.log10(np.maximum(maps['count'][observed], 1)).astype(np.float32)

shown_g1 = np.full(npix, hp.UNSEEN, dtype=np.float32)
shown_g2 = np.full(npix, hp.UNSEEN, dtype=np.float32)
for shown, key in [(shown_g1, 'gamma1'), (shown_g2, 'gamma2_namaster')]:
    vals = maps[key][observed]
    scale = np.nanpercentile(np.abs(vals), 98.0)
    shown[observed] = np.clip(vals, -scale, scale).astype(np.float32)

plt.figure(figsize=(13, 8))
hp.mollview(shown_mask, title='Tomo 4 normalized weight mask', sub=(2, 2, 1), unit='mean=1 on footprint')
hp.mollview(shown_log_count, title='Tomo 4 log10 source count', sub=(2, 2, 2), unit='log10(count)')
hp.mollview(shown_g1, title='Tomo 4 gamma1 clipped at 98 pct', sub=(2, 2, 3), unit='shear')
hp.mollview(shown_g2, title='Tomo 4 gamma2_namaster clipped at 98 pct', sub=(2, 2, 4), unit='shear')
plt.savefig(quicklook_path, dpi=170, bbox_inches='tight')
plt.close('all')

summary['quicklook'] = {'tomo4_shear_maps_quicklook': str(quicklook_path.relative_to(PACKAGE_ROOT))}
save_json(OUTDIR / 'summary.json', summary)
print('Wrote', quicklook_path)

Wrote /global/cfs/cdirs/lsst/www/shivamp/DESI/act_desi_ksz_transfer/diagnostics/des_y3_shear_tomo4_covariance/tomo4_shear_maps_quicklook.png


## 2. Reconstruct Tomo-4 Maps from the Processed DES Shear Pickle

This is the direct test of whether the HDF5 transfer map matches the processed catalog that was originally produced by `ACTxDES_measurements.ipynb`.

The comparison is against the NERSC provenance pickle path only. This path is not needed after transfer, but it is useful here on NERSC for diagnosing whether the transfer HDF5 was written correctly.

In [5]:
def residual_summary(reference, candidate, valid=None, rtol_floor=1e-12):
    ref = np.asarray(reference)
    cand = np.asarray(candidate)
    if valid is None:
        valid = np.ones(ref.shape, dtype=bool)
    else:
        valid = np.asarray(valid, dtype=bool)
    diff = cand[valid] - ref[valid]
    refv = ref[valid]
    finite = np.isfinite(diff) & np.isfinite(refv)
    diff = diff[finite]
    refv = refv[finite]
    if diff.size == 0:
        return {'n': 0}
    denom = np.maximum(np.abs(refv), rtol_floor)
    rel = np.abs(diff) / denom
    return {
        'n': int(diff.size),
        'max_abs': float(np.max(np.abs(diff))),
        'mean_abs': float(np.mean(np.abs(diff))),
        'rms_abs': float(np.sqrt(np.mean(diff**2))),
        'p50_abs': float(np.percentile(np.abs(diff), 50)),
        'p95_abs': float(np.percentile(np.abs(diff), 95)),
        'p99_abs': float(np.percentile(np.abs(diff), 99)),
        'max_rel': float(np.max(rel)),
        'p99_rel': float(np.percentile(rel, 99)),
    }


map_residuals = {
    'pickle_path_nersc_provenance': str(PICKLE_PATH),
    'ran': False,
}

catalog_arrays = None
if RUN_MAP_RECONSTRUCTION_FROM_PICKLE:
    if dill is None:
        raise RuntimeError('dill is required to load the processed DES shear pickle')
    if not PICKLE_PATH.exists():
        raise FileNotFoundError(PICKLE_PATH)

    with PICKLE_PATH.open('rb') as fp:
        cat = dill.load(fp)

    gamma1_cat, gamma2_cat, ra_deg, dec_deg, _placeholder, weight = cat[TOMO_INDEX]
    gamma1_cat = np.asarray(gamma1_cat)
    gamma2_cat = np.asarray(gamma2_cat)
    ra_deg = np.asarray(ra_deg)
    dec_deg = np.asarray(dec_deg)
    weight = np.asarray(weight)
    catalog_arrays = (gamma1_cat, gamma2_cat, ra_deg, dec_deg, weight)

    pix = hp.ang2pix(NSIDE, ra_deg, dec_deg, lonlat=True)
    count_re = np.bincount(pix, minlength=npix).astype(np.int64)
    sum_w = np.bincount(pix, weights=weight, minlength=npix)
    sum_w_g1 = np.bincount(pix, weights=weight * gamma1_cat, minlength=npix)
    sum_w_g2 = np.bincount(pix, weights=weight * gamma2_cat, minlength=npix)
    sum_w2 = np.bincount(pix, weights=weight**2, minlength=npix)
    sum_w2_e2_over2 = np.bincount(
        pix,
        weights=0.5 * weight**2 * (gamma1_cat**2 + gamma2_cat**2),
        minlength=npix,
    )

    obs_re = sum_w > 0
    gamma1_re = np.zeros(npix, dtype=np.float64)
    gamma2_re = np.zeros(npix, dtype=np.float64)
    gamma1_re[obs_re] = sum_w_g1[obs_re] / sum_w[obs_re]
    gamma2_re[obs_re] = sum_w_g2[obs_re] / sum_w[obs_re]
    gamma2_namaster_re = -gamma2_re
    mask_weight_re = np.zeros(npix, dtype=np.float64)
    mask_weight_re[obs_re] = sum_w[obs_re] / np.mean(sum_w[obs_re])
    mask_binary_re = obs_re.astype(np.float64)

    map_residuals.update({
        'ran': True,
        'n_sources_pickle': int(gamma1_cat.size),
        'n_observed_pixels_recomputed': int(np.count_nonzero(obs_re)),
        'mean_weight_per_observed_pixel_recomputed': float(np.mean(sum_w[obs_re])),
        'sum_weight_recomputed': float(np.sum(sum_w)),
        'sum_weight_hdf5': float(np.sum(maps['mask_weight_raw'])),
        'count': residual_summary(maps['count'].astype(np.int64), count_re, None),
        'mask_weight_raw': residual_summary(maps['mask_weight_raw'], sum_w, None),
        'mask_weight': residual_summary(maps['mask_weight'], mask_weight_re, observed | obs_re),
        'mask_binary': residual_summary(maps['mask_binary'], mask_binary_re, None),
        'gamma1_observed_union': residual_summary(maps['gamma1'], gamma1_re, observed | obs_re),
        'gamma2_namaster_observed_union': residual_summary(maps['gamma2_namaster'], gamma2_namaster_re, observed | obs_re),
        'sum_weight_sq': residual_summary(maps['sum_weight_sq'], sum_w2, None),
        'sum_w2_e2_over2': residual_summary(maps['sum_w2_e2_over2'], sum_w2_e2_over2, None),
    })

    del pix, count_re, sum_w, sum_w_g1, sum_w_g2, sum_w2, sum_w2_e2_over2
    del gamma1_re, gamma2_re, gamma2_namaster_re, mask_weight_re, mask_binary_re
else:
    summary['notes'].append('Skipped map reconstruction from pickle because RUN_MAP_RECONSTRUCTION_FROM_PICKLE=False')

summary['map_residuals_tomo4'] = map_residuals
save_json(OUTDIR / 'map_residuals_tomo4.json', map_residuals)
save_json(OUTDIR / 'summary.json', summary)
print(json.dumps(to_builtin(map_residuals), indent=2, sort_keys=True)[:5000])

{
  "count": {
    "max_abs": 0.0,
    "max_rel": 0.0,
    "mean_abs": 0.0,
    "n": 12582912,
    "p50_abs": 0.0,
    "p95_abs": 0.0,
    "p99_abs": 0.0,
    "p99_rel": 0.0,
    "rms_abs": 0.0
  },
  "gamma1_observed_union": {
    "max_abs": 5.912734946100784e-08,
    "max_rel": 5.9534251078282043e-08,
    "mean_abs": 1.5648055002827044e-09,
    "n": 1443154,
    "p50_abs": 9.041089005928926e-10,
    "p95_abs": 5.501925397222161e-09,
    "p99_abs": 7.9228313754065e-09,
    "p99_rel": 5.286823169516271e-08,
    "rms_abs": 2.493127327391181e-09
  },
  "gamma2_namaster_observed_union": {
    "max_abs": 4.877779558043471e-08,
    "max_rel": 5.95381566189276e-08,
    "mean_abs": 1.5626009501534055e-09,
    "n": 1443154,
    "p50_abs": 9.030943284860093e-10,
    "p95_abs": 5.499698167710231e-09,
    "p99_abs": 7.837807858290793e-09,
    "p99_rel": 5.283413035611003e-08,
    "rms_abs": 2.485765995888542e-09
  },
  "mask_binary": {
    "max_abs": 0.0,
    "max_rel": 0.0,
    "mean_abs": 0.0,


## 3. Optional Long Base-Catalog Audit

This checks the base DES Y3 index file without modifying it. It is intentionally optional because the raw catalog is large and the index file uses external links.

Use this only if the HDF5-vs-pickle comparison fails or if you need to verify the exact metacalibration selection counts against the original catalog machinery.

In [6]:
base_audit = {
    'ran': False,
    'base_indexcat_nersc_provenance': str(BASE_INDEXCAT),
    'note': 'This cell reads metadata/index information only and does not modify the base catalog.',
}

if RUN_BASE_CATALOG_AUDIT:
    if not BASE_INDEXCAT.exists():
        raise FileNotFoundError(BASE_INDEXCAT)
    with h5py.File(BASE_INDEXCAT, 'r') as h5:
        base_audit['ran'] = True
        base_audit['root_keys'] = list(h5.keys())
        base_audit['index_keys_matching_bin4'] = sorted([k for k in h5['index'].keys() if 'bin4' in k or k == 'select_bin'])
        counts = {}
        for key in ['select_bin4', 'select_1p_bin4', 'select_1m_bin4', 'select_2p_bin4', 'select_2m_bin4']:
            if key in h5['index']:
                counts[key] = int(h5['index'][key].shape[0])
        base_audit['index_counts'] = counts
        links = {}
        for group_name in ['catalog/metacal', 'catalog/gold', 'catalog/sompz']:
            try:
                obj = h5[group_name]
                links[group_name] = {'resolves': True, 'shape_or_keys': list(obj.keys())[:10] if hasattr(obj, 'keys') else str(obj.shape)}
            except Exception as exc:
                links[group_name] = {'resolves': False, 'error': repr(exc)}
        base_audit['external_link_resolution'] = links
else:
    base_audit['reason_skipped'] = 'RUN_BASE_CATALOG_AUDIT=False'

summary['base_catalog_audit'] = base_audit
save_json(OUTDIR / 'summary.json', summary)
print(json.dumps(to_builtin(base_audit), indent=2, sort_keys=True)[:5000])

{
  "base_indexcat_nersc_provenance": "/global/cfs/cdirs/lsst/www/shivamp/data_actxdes/Y3_cats/DESY3_indexcat.h5",
  "note": "This cell reads metadata/index information only and does not modify the base catalog.",
  "ran": false,
  "reason_skipped": "RUN_BASE_CATALOG_AUDIT=False"
}


## 4. NaMaster Helpers

These helpers keep the mask/noise convention explicit. The installed NaMaster on this machine accepts `f_ell` through `NmtBin(...)`, but not through `NmtBin.from_edges(...)`; `make_nmt_bins` supports both APIs.

In [7]:
def make_nmt_bins(ell_left, ell_right, nside, pixwin_pol=None, apply_pixel_window=True):
    if nmt is None:
        raise RuntimeError('NaMaster is not available')
    ell_left = np.asarray(ell_left, dtype=np.int32)
    ell_right = np.asarray(ell_right, dtype=np.int32)
    lmax = int(ell_right[-1] - 1)
    f_ell = None
    if apply_pixel_window:
        if pixwin_pol is None:
            raise ValueError('pixwin_pol is required when apply_pixel_window=True')
        f_ell = np.ones(lmax + 1, dtype=np.float64)
        usable = np.asarray(pixwin_pol[:lmax + 1], dtype=np.float64)
        good = usable > 0
        f_ell[good] = 1.0 / usable[good]**2
        f_ell[~good] = 0.0

    try:
        if f_ell is None:
            return nmt.NmtBin.from_edges(ell_left, ell_right, is_Dell=False)
        return nmt.NmtBin.from_edges(ell_left, ell_right, is_Dell=False, f_ell=f_ell)
    except TypeError:
        ells = np.arange(lmax + 1, dtype=np.int32)
        bpws = -np.ones(lmax + 1, dtype=np.int32)
        weights = np.zeros(lmax + 1, dtype=np.float64)
        for ib, (lo, hi) in enumerate(zip(ell_left, ell_right)):
            lo = max(int(lo), 0)
            hi = min(int(hi), lmax + 1)
            bpws[lo:hi] = ib
            weights[lo:hi] = 1.0
        return nmt.NmtBin(
            nside=nside,
            ells=ells,
            bpws=bpws,
            weights=weights,
            lmax=lmax,
            is_Dell=False,
            f_ell=f_ell,
        )


def make_spin2_field(mask, gamma1, gamma2_namaster, lmax):
    mask = np.asarray(mask, dtype=np.float64)
    g1 = np.asarray(gamma1, dtype=np.float64)
    g2 = np.asarray(gamma2_namaster, dtype=np.float64)
    kwargs = {
        'spin': 2,
        'purify_e': False,
        'purify_b': False,
        'n_iter': 0,
        'lmax_sht': int(lmax),
        'lite': True,
    }
    try:
        return nmt.NmtField(mask, [g1, g2], **kwargs)
    except TypeError:
        kwargs.pop('lite', None)
        return nmt.NmtField(mask, [g1, g2], **kwargs)


def make_noise_template(noise_level, lmax):
    arr = np.zeros((4, int(lmax) + 1), dtype=np.float64)
    arr[0, :] = float(noise_level)  # EE
    arr[3, :] = float(noise_level)  # BB
    return arr


def decouple_with_noise(workspace, coupled_cell, noise_template):
    try:
        return workspace.decouple_cell(coupled_cell, cl_noise=noise_template)
    except TypeError:
        return workspace.decouple_cell(coupled_cell - noise_template)


def bandpowers_to_full(cl_bpw, ell_left, ell_right, lmax):
    cl_bpw = np.asarray(cl_bpw, dtype=np.float64)
    full = np.zeros((cl_bpw.shape[0], int(lmax) + 1), dtype=np.float64)
    for ib, (lo, hi) in enumerate(zip(ell_left, ell_right)):
        lo = max(int(lo), 0)
        hi = min(int(hi), int(lmax) + 1)
        full[:, lo:hi] = cl_bpw[:, ib][:, None]
    if int(ell_left[0]) > 0:
        full[:, :int(ell_left[0])] = cl_bpw[:, 0][:, None]
    return full


def sanitize_total_cls(cl_full, floor=None, zero_cross=True):
    out = np.nan_to_num(np.asarray(cl_full, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0).copy()
    if floor is None:
        positive = out[[0, 3], :]
        positive = positive[positive > 0]
        floor = float(np.min(positive) * 1e-6) if positive.size else 1e-20
    out[0, :] = np.maximum(out[0, :], floor)
    out[3, :] = np.maximum(out[3, :], floor)
    if zero_cross:
        out[1, :] = 0.0
        out[2, :] = 0.0
    return out


def bin_full_ell_covariance(ell_cov, ell_left, ell_right, lmax):
    """Compress a full-ell EE covariance matrix to the notebook bandpowers.

    NaMaster returns full coupled-ell covariance when gaussian_covariance is
    called with coupled=True. This binning is only for diagnostics so the
    coupled=True result can be plotted beside the decoupled bandpower cases.
    """
    ell_cov = np.asarray(ell_cov, dtype=np.float64)
    nbpw = len(ell_left)
    out = np.zeros((nbpw, nbpw), dtype=np.float64)
    for ib, (lo_i, hi_i) in enumerate(zip(ell_left, ell_right)):
        lo_i = max(int(lo_i), 0)
        hi_i = min(int(hi_i), int(lmax) + 1)
        if hi_i <= lo_i:
            continue
        wi = np.full(hi_i - lo_i, 1.0 / (hi_i - lo_i), dtype=np.float64)
        for jb, (lo_j, hi_j) in enumerate(zip(ell_left, ell_right)):
            lo_j = max(int(lo_j), 0)
            hi_j = min(int(hi_j), int(lmax) + 1)
            if hi_j <= lo_j:
                continue
            wj = np.full(hi_j - lo_j, 1.0 / (hi_j - lo_j), dtype=np.float64)
            out[ib, jb] = wi @ ell_cov[lo_i:hi_i, lo_j:hi_j] @ wj
    return out


def extract_ee_covariance(cov, nbpw, ncls=4, ell_left=None, ell_right=None, lmax=None):
    arr = np.asarray(cov)
    if arr.ndim == 4:
        return arr[:, 0, :, 0]
    if arr.ndim != 2:
        raise ValueError(f'Unexpected covariance shape {arr.shape}')

    expected_bpw = (nbpw * ncls, nbpw * ncls)
    if arr.shape == expected_bpw:
        return arr.reshape(nbpw, ncls, nbpw, ncls)[:, 0, :, 0]

    if arr.shape[0] == arr.shape[1] and arr.shape[0] % ncls == 0:
        nell = arr.shape[0] // ncls
        if ell_left is None or ell_right is None:
            raise ValueError(
                f'Got full-ell covariance shape {arr.shape}; pass ell_left/ell_right to bin it to bandpowers.'
            )
        if lmax is None:
            lmax = nell - 1
        if nell != int(lmax) + 1:
            raise ValueError(f'Full-ell covariance has nell={nell}, but lmax+1={int(lmax) + 1}')
        ee_ell_cov = arr.reshape(nell, ncls, nell, ncls)[:, 0, :, 0]
        return bin_full_ell_covariance(ee_ell_cov, ell_left, ell_right, lmax)

    raise ValueError(f'Unexpected covariance shape {arr.shape}; expected {expected_bpw} or full-ell ncls blocks')


def corr_from_cov(cov):
    diag = np.diag(cov)
    denom = np.sqrt(np.outer(np.maximum(diag, 0), np.maximum(diag, 0)))
    corr = np.zeros_like(cov)
    good = denom > 0
    corr[good] = cov[good] / denom[good]
    return corr

print('NaMaster helper functions defined. nmt available:', nmt is not None)

NaMaster helper functions defined. nmt available: True


## 5. Spectrum Invariance Across Mask Normalizations

A constant rescaling of the weighted mask should not change the decoupled signal spectrum if the matching shape-noise pseudo-Cl is used. This cell compares the current raw-mask convention with the normalized weighted mask and a binary-mask control.

In [9]:
spectrum_results = {}

if RUN_NAMASTER_SPECTRA:
    if nmt is None:
        raise RuntimeError('RUN_NAMASTER_SPECTRA=True but pymaster is not available')

    lmax = int(ell_right[-1] - 1)
    bins = make_nmt_bins(ell_left, ell_right, NSIDE, pixwin_pol=pixwin_pol, apply_pixel_window=True)
    ell_eff = bins.get_effective_ells()

    scenarios = {
        'raw_weight_mask': {
            'mask_name': 'mask_weight_raw',
            'noise_level': raw_noise,
            'description': 'Raw weighted count mask, sum_i w_i per pixel',
        },
        'normalized_weight_mask': {
            'mask_name': 'mask_weight',
            'noise_level': norm_noise,
            'description': 'Raw weighted count mask divided by mean observed-pixel weight',
        },
        'binary_mask': {
            'mask_name': 'mask_binary',
            'noise_level': binary_noise,
            'description': 'Binary occupied-pixel mask control',
        },
    }

    for label, cfg in scenarios.items():
        print('Measuring scenario:', label)
        field = make_spin2_field(
            maps[cfg['mask_name']],
            maps['gamma1'],
            maps['gamma2_namaster'],
            lmax=lmax,
        )
        workspace = nmt.NmtWorkspace()
        workspace.compute_coupling_matrix(field, field, bins, n_iter=0, lmax_mask=lmax)
        coupled = nmt.compute_coupled_cell(field, field)
        noise_template = make_noise_template(cfg['noise_level'], lmax)
        cl_no_noise = decouple_with_noise(workspace, coupled, noise_template)
        cl_subtract_first = workspace.decouple_cell(coupled - noise_template)
        cl_total_decoupled = workspace.decouple_cell(coupled)
        noise_decoupled = workspace.decouple_cell(noise_template)

        spectrum_results[label] = {
            'field': field,
            'workspace': workspace,
            'coupled': coupled,
            'noise_template': noise_template,
            'noise_decoupled': noise_decoupled,
            'cl_noise_subtracted': cl_no_noise,
            'cl_subtract_first': cl_subtract_first,
            'cl_total_decoupled': cl_total_decoupled,
            'mask_name': cfg['mask_name'],
            'noise_level': float(cfg['noise_level']),
            'description': cfg['description'],
        }

    raw_ee = spectrum_results['raw_weight_mask']['cl_noise_subtracted'][0]
    norm_ee = spectrum_results['normalized_weight_mask']['cl_noise_subtracted'][0]
    binary_ee = spectrum_results['binary_mask']['cl_noise_subtracted'][0]
    denom = np.maximum(np.abs(raw_ee), 1e-30)
    spectrum_summary = {
        'ran': True,
        'lmax': lmax,
        'n_bandpowers': int(len(ell_eff)),
        'ell_eff': ell_eff.tolist(),
        'raw_vs_normalized_ee_max_abs': float(np.max(np.abs(raw_ee - norm_ee))),
        'raw_vs_normalized_ee_p99_frac': float(np.percentile(np.abs(raw_ee - norm_ee) / denom, 99)),
        'raw_vs_binary_ee_p99_frac': float(np.percentile(np.abs(raw_ee - binary_ee) / denom, 99)),
        'noise_subtraction_path_max_abs_raw': float(np.max(np.abs(
            spectrum_results['raw_weight_mask']['cl_noise_subtracted'] - spectrum_results['raw_weight_mask']['cl_subtract_first']
        ))),
        'first_five_raw_ee_noise_subtracted': raw_ee[:5].tolist(),
        'first_five_raw_bb_noise_subtracted': spectrum_results['raw_weight_mask']['cl_noise_subtracted'][3, :5].tolist(),
    }
else:
    ell_eff = None
    lmax = int(ell_right[-1] - 1)
    spectrum_summary = {'ran': False, 'reason_skipped': 'RUN_NAMASTER_SPECTRA=False'}

summary['spectrum_invariance'] = spectrum_summary
save_json(OUTDIR / 'summary.json', summary)
print(json.dumps(to_builtin(spectrum_summary), indent=2, sort_keys=True)[:5000])

Measuring scenario: raw_weight_mask
Measuring scenario: normalized_weight_mask
Measuring scenario: binary_mask
{
  "ell_eff": [
    12.0,
    23.0,
    37.5,
    55.50000000000001,
    76.99999999999999,
    102.00000000000001,
    131.0,
    163.00000000000009,
    198.5,
    238.0,
    280.5,
    326.5,
    376.5,
    429.99999999999994,
    486.5,
    546.5,
    610.5,
    678.0000000000003,
    748.5,
    822.5,
    900.5,
    981.9999999999999,
    1066.4999999999998,
    1154.9999999999998,
    1246.9999999999998,
    1342.0000000000005,
    1441.0000000000002,
    1543.5,
    1649.5,
    1758.9999999999998,
    1872.0000000000005,
    1988.9999999999998
  ],
  "first_five_raw_bb_noise_subtracted": [
    5.824743155355385e-09,
    -5.710212446979867e-10,
    -4.677615002330851e-10,
    1.8068244050718412e-10,
    8.647453882920937e-10
  ],
  "first_five_raw_ee_noise_subtracted": [
    1.2476017279205766e-08,
    2.6330028414302483e-08,
    1.830895168168983e-08,
    1.07355663083

In [10]:
# Spectrum-only comparison plot, before covariance error bars.
spectrum_plot_path = OUTDIR / 'tomo4_spectrum_mask_invariance.png'

if spectrum_results:
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, result in spectrum_results.items():
        cl = result['cl_noise_subtracted']
        ax.plot(ell_eff, ell_eff * cl[0] * 1e7, marker='o', ms=3, lw=1.2, label=label.replace('_', ' '))
    ax.axhline(0, color='0.3', lw=0.8)
    ax.set_xlabel(r'$\ell$')
    ax.set_ylabel(r'$\ell C_\ell^{EE} \times 10^7$')
    ax.set_title('DES Y3 tomo 4 auto spectrum: mask/noise convention check')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    fig.savefig(spectrum_plot_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
else:
    write_placeholder_plot(spectrum_plot_path, 'Spectrum check skipped', 'Set RUN_NAMASTER_SPECTRA=True and rerun the notebook.')

summary.setdefault('plots', {})['tomo4_spectrum_mask_invariance'] = str(spectrum_plot_path.relative_to(PACKAGE_ROOT))
save_json(OUTDIR / 'summary.json', summary)
print('Wrote', spectrum_plot_path)

Wrote /global/cfs/cdirs/lsst/www/shivamp/DESI/act_desi_ksz_transfer/diagnostics/des_y3_shear_tomo4_covariance/tomo4_spectrum_mask_invariance.png


## 6. Shape-Noise Handling and Random-Rotation Check

The first part compares two equivalent NaMaster subtraction paths. The second part, if enabled, random-rotates catalog ellipticities and compares the resulting noise spectrum against the stored analytic pseudo-noise level.

`RUN_RANDOM_ROTATIONS` defaults to `True` with 20 rotations, matching the diagnostic plan. Set it to `False` for a faster smoke run.

In [9]:
noise_check = {
    'ran_subtraction_path_check': False,
    'ran_random_rotations': False,
}

noise_plot_path = OUTDIR / 'tomo4_noise_rotation_check.png'

if spectrum_results:
    raw_result = spectrum_results['raw_weight_mask']
    diff = raw_result['cl_noise_subtracted'] - raw_result['cl_subtract_first']
    noise_check.update({
        'ran_subtraction_path_check': True,
        'raw_noise_level_pseudo': raw_noise,
        'raw_noise_decoupled_first_five_ee': raw_result['noise_decoupled'][0, :5].tolist(),
        'max_abs_difference_cl_noise_kwarg_vs_subtract_first': float(np.max(np.abs(diff))),
        'p99_abs_difference_cl_noise_kwarg_vs_subtract_first': float(np.percentile(np.abs(diff), 99)),
    })

if RUN_RANDOM_ROTATIONS:
    if not spectrum_results:
        raise RuntimeError('Random rotations require RUN_NAMASTER_SPECTRA=True so the raw workspace exists')
    if dill is None:
        raise RuntimeError('dill is required for random rotations from the processed pickle')
    if catalog_arrays is None:
        with PICKLE_PATH.open('rb') as fp:
            cat = dill.load(fp)
        gamma1_cat, gamma2_cat, ra_deg, dec_deg, _placeholder, weight = cat[TOMO_INDEX]
        catalog_arrays = tuple(np.asarray(x) for x in [gamma1_cat, gamma2_cat, ra_deg, dec_deg, weight])

    gamma1_cat, gamma2_cat, ra_deg, dec_deg, weight = catalog_arrays
    pix = hp.ang2pix(NSIDE, ra_deg, dec_deg, lonlat=True)
    sum_w = np.bincount(pix, weights=weight, minlength=npix)
    obs_re = sum_w > 0
    raw_mask = maps['mask_weight_raw']
    raw_workspace = spectrum_results['raw_weight_mask']['workspace']
    raw_noise_decoupled_ee = spectrum_results['raw_weight_mask']['noise_decoupled'][0]
    rng = np.random.default_rng(RANDOM_SEED)
    rotated_ee = []

    for irot in range(N_RANDOM_ROTATIONS):
        phi = rng.uniform(0.0, 2.0 * np.pi, size=gamma1_cat.size)
        c2 = np.cos(2.0 * phi)
        s2 = np.sin(2.0 * phi)
        g1_rot = gamma1_cat * c2 - gamma2_cat * s2
        g2_rot = gamma1_cat * s2 + gamma2_cat * c2
        sum_w_g1 = np.bincount(pix, weights=weight * g1_rot, minlength=npix)
        sum_w_g2 = np.bincount(pix, weights=weight * g2_rot, minlength=npix)
        map_g1 = np.zeros(npix, dtype=np.float64)
        map_g2_nmt = np.zeros(npix, dtype=np.float64)
        map_g1[obs_re] = sum_w_g1[obs_re] / sum_w[obs_re]
        map_g2_nmt[obs_re] = -sum_w_g2[obs_re] / sum_w[obs_re]
        frot = make_spin2_field(raw_mask, map_g1, map_g2_nmt, lmax=lmax)
        pcl_rot = nmt.compute_coupled_cell(frot, frot)
        rotated_ee.append(raw_workspace.decouple_cell(pcl_rot)[0])
        print(f'Finished random rotation {irot + 1}/{N_RANDOM_ROTATIONS}')

    rotated_ee = np.asarray(rotated_ee)
    mean_rot = np.mean(rotated_ee, axis=0)
    std_rot = np.std(rotated_ee, axis=0, ddof=1) if N_RANDOM_ROTATIONS > 1 else np.zeros_like(mean_rot)
    ratio = mean_rot / np.maximum(raw_noise_decoupled_ee, 1e-30)
    noise_check.update({
        'ran_random_rotations': True,
        'n_random_rotations': int(N_RANDOM_ROTATIONS),
        'random_seed': int(RANDOM_SEED),
        'mean_rotation_over_analytic_noise_first_five': ratio[:5].tolist(),
        'median_rotation_over_analytic_noise': float(np.median(ratio[np.isfinite(ratio)])),
        'p16_p84_rotation_over_analytic_noise': np.percentile(ratio[np.isfinite(ratio)], [16, 84]).tolist(),
    })

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(ell_eff, ell_eff * mean_rot * 1e7, yerr=ell_eff * std_rot * 1e7, fmt='o', ms=3, lw=1, label='random rotations')
    ax.plot(ell_eff, ell_eff * raw_noise_decoupled_ee * 1e7, color='k', lw=1.5, label='analytic decoupled noise')
    ax.set_xlabel(r'$\ell$')
    ax.set_ylabel(r'$\ell N_\ell^{EE} \times 10^7$')
    ax.set_title('Tomo 4 random-rotation shape-noise check')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    fig.savefig(noise_plot_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
else:
    if not RUN_RANDOM_ROTATIONS:
        noise_check['reason_random_rotations_skipped'] = 'RUN_RANDOM_ROTATIONS=False; set True for final acceptance run.'
    write_placeholder_plot(
        noise_plot_path,
        'Random-rotation check skipped',
        'Set RUN_RANDOM_ROTATIONS=True. The analytic subtraction-path check still ran if NaMaster spectra were enabled.',
    )

summary['shape_noise_check'] = noise_check
summary.setdefault('plots', {})['tomo4_noise_rotation_check'] = str(noise_plot_path.relative_to(PACKAGE_ROOT))
save_json(OUTDIR / 'summary.json', summary)
print(json.dumps(to_builtin(noise_check), indent=2, sort_keys=True)[:5000])
print('Wrote', noise_plot_path)

Finished random rotation 1/20
Finished random rotation 2/20
Finished random rotation 3/20
Finished random rotation 4/20
Finished random rotation 5/20
Finished random rotation 6/20
Finished random rotation 7/20
Finished random rotation 8/20
Finished random rotation 9/20
Finished random rotation 10/20
Finished random rotation 11/20
Finished random rotation 12/20
Finished random rotation 13/20
Finished random rotation 14/20
Finished random rotation 15/20
Finished random rotation 16/20
Finished random rotation 17/20
Finished random rotation 18/20
Finished random rotation 19/20
Finished random rotation 20/20
{
  "max_abs_difference_cl_noise_kwarg_vs_subtract_first": 0.0,
  "mean_rotation_over_analytic_noise_first_five": [
    1.0733744041205815,
    0.9324816645152917,
    1.015986135365819,
    1.024770623187937,
    0.9903199042661331
  ],
  "median_rotation_over_analytic_noise": 1.0011702039225123,
  "n_random_rotations": 20,
  "p16_p84_rotation_over_analytic_noise": [
    0.996360828758

## 7. NaMaster Gaussian Covariance Variants

This is the central diagnostic. It compares:

- `current_suspect_pseudo_div_mean_mask2_uncoupled`: the problematic pattern, using a coupled pseudo-spectrum divided by a scalar mask moment but asking NaMaster for decoupled covariance.
- `corrected_data_decoupled_total`: a data-driven decoupled total spectrum, built as measured signal plus decoupled analytic noise.
- `smoothed_decoupled_total`: a smoothed positive version of the data-driven total spectrum, closer to what one would pass from theory.
- `coupled_total_data`: the actual coupled pseudo-spectrum, but with `coupled=True`.

If the first variant is the outlier with much larger diagonal errors, the problem is the covariance input convention rather than the shear map.

In [11]:
def smooth_positive_bandpowers(y, floor=1e-20):
    y = np.asarray(y, dtype=np.float64)
    yp = np.maximum(y, floor)
    if gaussian_filter1d is None or y.size < 5:
        return yp
    return np.exp(gaussian_filter1d(np.log(yp), sigma=1.0, mode='nearest'))


covariance_results = {}
mode_count_check = {}

if RUN_NAMASTER_COVARIANCE:
    if not spectrum_results:
        raise RuntimeError('RUN_NAMASTER_COVARIANCE=True requires RUN_NAMASTER_SPECTRA=True')

    raw_result = spectrum_results['raw_weight_mask']
    raw_field = raw_result['field']
    raw_workspace = raw_result['workspace']
    raw_mask = maps['mask_weight_raw'].astype(np.float64)
    nbpw = len(ell_eff)

    cov_workspace = nmt.NmtCovarianceWorkspace()
    cov_workspace.compute_coupling_coefficients(raw_field, raw_field, raw_field, raw_field, lmax=lmax, n_iter=0)

    current_suspect = raw_result['coupled'] / np.mean(raw_mask**2)

    cl_signal_bpw = raw_result['cl_noise_subtracted']
    cl_noise_bpw = raw_result['noise_decoupled']
    cl_total_bpw = cl_signal_bpw + cl_noise_bpw
    cl_total_bpw[1, :] = 0.0
    cl_total_bpw[2, :] = 0.0
    cl_total_full = sanitize_total_cls(bandpowers_to_full(cl_total_bpw, ell_left, ell_right, lmax))

    smoothed_bpw = np.zeros_like(cl_total_bpw)
    smoothed_bpw[0, :] = smooth_positive_bandpowers(cl_total_bpw[0, :])
    smoothed_bpw[3, :] = smooth_positive_bandpowers(np.maximum(cl_total_bpw[3, :], cl_noise_bpw[3, :]))
    smoothed_full = sanitize_total_cls(bandpowers_to_full(smoothed_bpw, ell_left, ell_right, lmax))

    covariance_inputs = {
        'current_suspect_pseudo_div_mean_mask2_uncoupled': {
            'cl': sanitize_total_cls(current_suspect),
            'coupled': False,
            'description': 'Coupled pseudo-Cl divided by mean(mask^2), then used as if uncoupled.',
        },
        'corrected_data_decoupled_total': {
            'cl': cl_total_full,
            'coupled': False,
            'description': 'Decoupled measured signal plus decoupled analytic noise, expanded to ell.',
        },
        'smoothed_decoupled_total': {
            'cl': smoothed_full,
            'coupled': False,
            'description': 'Smoothed positive theory-like total Cl, expanded from bandpowers.',
        },
        'coupled_total_data': {
            'cl': sanitize_total_cls(raw_result['coupled'], zero_cross=False),
            'coupled': True,
            'description': 'Actual coupled pseudo-Cl with coupled=True.',
        },
    }

    for label, cfg in covariance_inputs.items():
        print('Computing covariance variant:', label)
        cov = nmt.gaussian_covariance(
            cov_workspace,
            2,
            2,
            2,
            2,
            cfg['cl'],
            cfg['cl'],
            cfg['cl'],
            cfg['cl'],
            raw_workspace,
            raw_workspace,
            coupled=cfg['coupled'],
        )
        cov_shape = tuple(np.asarray(cov).shape)
        cov_space = 'full_ell_coupled' if cov_shape == (4 * (lmax + 1), 4 * (lmax + 1)) else 'bandpower_decoupled'
        ee_cov = extract_ee_covariance(cov, nbpw, ncls=4, ell_left=ell_left, ell_right=ell_right, lmax=lmax)
        ee_diag = np.diag(ee_cov)
        yerr = np.sqrt(np.maximum(ee_diag, 0.0))
        covariance_results[label] = {
            'description': cfg['description'],
            'coupled': bool(cfg['coupled']),
            'covariance_shape': cov_shape,
            'covariance_space': cov_space,
            'ee_cov': ee_cov,
            'ee_yerr': yerr,
            'ee_diag': ee_diag,
            'n_negative_diag': int(np.count_nonzero(ee_diag < 0)),
            'first_five_yerr': yerr[:5].tolist(),
        }

    delta_ell = ell_right - ell_left
    fsky_eff = float(np.mean(raw_mask)**2 / np.mean(raw_mask**2))
    total_ee_bpw = cl_total_bpw[0, :]
    mode_count_yerr = np.sqrt(2.0 / np.maximum((2.0 * ell_eff + 1.0) * delta_ell * fsky_eff, 1e-30)) * np.abs(total_ee_bpw)
    mode_count_check = {
        'fsky_eff_mean_mask_sq_over_mean_mask2': fsky_eff,
        'first_five_mode_count_yerr': mode_count_yerr[:5].tolist(),
    }

    cov_summary = {
        'ran': True,
        'variants': {
            label: {
                'description': res['description'],
                'coupled': res['coupled'],
                'covariance_shape': list(res['covariance_shape']),
                'covariance_space': res['covariance_space'],
                'n_negative_diag': res['n_negative_diag'],
                'first_five_yerr': res['first_five_yerr'],
            }
            for label, res in covariance_results.items()
        },
        'mode_count_check': mode_count_check,
    }
    if 'current_suspect_pseudo_div_mean_mask2_uncoupled' in covariance_results and 'corrected_data_decoupled_total' in covariance_results:
        ys = covariance_results['current_suspect_pseudo_div_mean_mask2_uncoupled']['ee_yerr']
        yc = covariance_results['corrected_data_decoupled_total']['ee_yerr']
        cov_summary['suspect_over_corrected_yerr_ratio_median'] = float(np.median(ys / np.maximum(yc, 1e-30)))
        cov_summary['suspect_over_corrected_yerr_ratio_first_five'] = (ys[:5] / np.maximum(yc[:5], 1e-30)).tolist()
else:
    cov_summary = {'ran': False, 'reason_skipped': 'RUN_NAMASTER_COVARIANCE=False'}

summary['covariance_diagnostics'] = cov_summary
save_json(OUTDIR / 'summary.json', summary)
print(json.dumps(to_builtin(cov_summary), indent=2, sort_keys=True)[:5000])

Computing covariance variant: current_suspect_pseudo_div_mean_mask2_uncoupled
Computing covariance variant: corrected_data_decoupled_total
Computing covariance variant: smoothed_decoupled_total
Computing covariance variant: coupled_total_data
{
  "mode_count_check": {
    "first_five_mode_count_yerr": [
      5.804980286494414e-09,
      6.052343618331616e-09,
      3.209450180408857e-09,
      1.6100430785111241e-09,
      9.724918543114876e-10
    ],
    "fsky_eff_mean_mask_sq_over_mean_mask2": 0.08998836636932507
  },
  "ran": true,
  "suspect_over_corrected_yerr_ratio_first_five": [
    0.8407983861099166,
    0.7144304151348845,
    0.7584592706445656,
    0.839156331978624,
    0.8612427577290971
  ],
  "suspect_over_corrected_yerr_ratio_median": 0.8546044230637164,
  "variants": {
    "corrected_data_decoupled_total": {
      "coupled": false,
      "covariance_shape": [
        128,
        128
      ],
      "covariance_space": "bandpower_decoupled",
      "description": "Deco

In [12]:
cls_cov_plot_path = OUTDIR / 'tomo4_cls_covariance_variants.png'
corr_plot_path = OUTDIR / 'tomo4_covariance_correlation_matrices.png'

if covariance_results:
    raw_cl = spectrum_results['raw_weight_mask']['cl_noise_subtracted'][0]
    comparable_labels = [
        label for label, res in covariance_results.items()
        if res.get('covariance_space') == 'bandpower_decoupled'
    ]
    skipped_labels = [
        label for label, res in covariance_results.items()
        if res.get('covariance_space') != 'bandpower_decoupled'
    ]

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for label in comparable_labels:
        res = covariance_results[label]
        pretty = label.replace('_', ' ')
        ax.errorbar(
            ell_eff,
            ell_eff * raw_cl * 1e7,
            yerr=ell_eff * res['ee_yerr'] * 1e7,
            fmt='o',
            ms=3,
            lw=1,
            capsize=1.5,
            label=pretty,
            alpha=0.85,
        )
    if mode_count_check:
        ax.plot(ell_eff, ell_eff * raw_cl * 1e7, color='k', lw=1.0, alpha=0.5, label='measured EE')
    ax.axhline(0, color='0.3', lw=0.8)
    ax.set_xlabel(r'$\ell$')
    ax.set_ylabel(r'$\ell C_\ell^{EE} \times 10^7$')
    ax.set_title('DES Y3 tomo 4 auto spectrum: decoupled covariance variants')
    if skipped_labels:
        ax.text(
            0.02,
            0.02,
            'Skipped full-ell coupled covariance in this plot: ' + ', '.join(skipped_labels),
            transform=ax.transAxes,
            fontsize=7,
            va='bottom',
            ha='left',
            bbox={'facecolor': 'white', 'alpha': 0.75, 'edgecolor': '0.8'},
        )
    ax.legend(fontsize=7)
    ax.grid(alpha=0.25)
    fig.savefig(cls_cov_plot_path, dpi=180, bbox_inches='tight')
    plt.close(fig)

    labels = list(covariance_results.keys())
    ncols = min(2, len(labels))
    nrows = int(np.ceil(len(labels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.8 * nrows), squeeze=False)
    for ax, label in zip(axes.flat, labels):
        corr = corr_from_cov(covariance_results[label]['ee_cov'])
        im = ax.imshow(corr, origin='lower', vmin=-1, vmax=1, cmap='coolwarm')
        ax.set_title(label.replace('_', ' '), fontsize=9)
        ax.set_xlabel('bandpower')
        ax.set_ylabel('bandpower')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for ax in axes.flat[len(labels):]:
        ax.axis('off')
    fig.suptitle('EE x EE covariance correlation matrices', y=0.995)
    fig.tight_layout()
    fig.savefig(corr_plot_path, dpi=170, bbox_inches='tight')
    plt.close(fig)
else:
    write_placeholder_plot(cls_cov_plot_path, 'Covariance variants skipped', 'Set RUN_NAMASTER_COVARIANCE=True and RUN_NAMASTER_SPECTRA=True.')
    write_placeholder_plot(corr_plot_path, 'Covariance correlations skipped', 'Set RUN_NAMASTER_COVARIANCE=True and RUN_NAMASTER_SPECTRA=True.')

summary.setdefault('plots', {})['tomo4_cls_covariance_variants'] = str(cls_cov_plot_path.relative_to(PACKAGE_ROOT))
summary.setdefault('plots', {})['tomo4_covariance_correlation_matrices'] = str(corr_plot_path.relative_to(PACKAGE_ROOT))
save_json(OUTDIR / 'summary.json', summary)
print('Wrote', cls_cov_plot_path)
print('Wrote', corr_plot_path)

Wrote /global/cfs/cdirs/lsst/www/shivamp/DESI/act_desi_ksz_transfer/diagnostics/des_y3_shear_tomo4_covariance/tomo4_cls_covariance_variants.png
Wrote /global/cfs/cdirs/lsst/www/shivamp/DESI/act_desi_ksz_transfer/diagnostics/des_y3_shear_tomo4_covariance/tomo4_covariance_correlation_matrices.png


## 8. Paper-Style Corrected Covariance Plot

This is the comparison plot to use against the `4,4` panel of Fig. 4 in the DES Y3 harmonic-space paper.  It intentionally uses only a decoupled bandpower covariance (`coupled=False`) and defaults to the smoothed decoupled total spectrum as the covariance input.  The full-ell `coupled=True` covariance diagnostic is not comparable to this plot.

In [15]:
import pymaster
print(pymaster.__version__)


AttributeError: module 'pymaster' has no attribute '__version__'

In [ ]:
paper_style_plot_path = OUTDIR / 'tomo4_cls_paper_style_corrected_covariance.png'
paper_covariance_label = 'smoothed_decoupled_total'
if paper_covariance_label not in covariance_results:
    paper_covariance_label = 'corrected_data_decoupled_total'
if paper_covariance_label not in covariance_results:
    raise RuntimeError('No decoupled covariance result is available; rerun the covariance diagnostics cell first.')

paper_cov = covariance_results[paper_covariance_label]
if paper_cov.get('covariance_space') != 'bandpower_decoupled':
    raise RuntimeError(f'{paper_covariance_label} is not a decoupled bandpower covariance: {paper_cov.get("covariance_space")}')

raw_cl = spectrum_results['raw_weight_mask']['cl_noise_subtracted']
y_ee = ell_eff * raw_cl[0] * 1e7
yerr_ee = ell_eff * paper_cov['ee_yerr'] * 1e7
y_bb = ell_eff * raw_cl[3] * 1e7

fig = plt.figure(figsize=(12.2, 4.5))
grid = fig.add_gridspec(1, 2, width_ratios=[1.1, 1.0], wspace=0.18)
ax = fig.add_subplot(grid[0, 0])

x = np.sqrt(ell_eff)
ax.errorbar(
    x,
    y_ee,
    yerr=yerr_ee,
    fmt='o',
    ms=4.2,
    color='#1266c3',
    ecolor='#1266c3',
    capsize=0,
    lw=1.4,
    label='DES Y3 tomo 4 x 4',
)
ax.plot(x, y_bb, color='0.55', lw=1.0, alpha=0.85, label='BB noise-subtracted')
ax.axhline(0, color='0.2', lw=0.8, ls='--')

# Approximate DES-Y3-style scale-cut bands for visual comparison only.
for lo, hi, alpha in [(200, 300, 0.10), (400, 2048, 0.18)]:
    ax.axvspan(np.sqrt(lo), np.sqrt(hi), color='0.5', alpha=alpha, zorder=0)

xticks = np.array([0, 100, 400, 900, 1600], dtype=float)
ax.set_xticks(np.sqrt(xticks))
ax.set_xticklabels([str(int(t)) for t in xticks])
ax.set_xlim(0, np.sqrt(2048) * 1.02)
ax.set_ylim(-1.2, 8.6)
ax.set_xlabel(r'Multipole $\ell$')
ax.set_ylabel(r'$\ell C_\ell^{EE}\ (10^{-7})$')
ax.set_title('This measurement: corrected decoupled covariance')
ax.text(0.03, 0.95, '4,4', transform=ax.transAxes, ha='left', va='top', fontsize=13)
ax.text(
    0.03,
    0.84,
    f'covariance: {paper_covariance_label.replace("_", " ")}',
    transform=ax.transAxes,
    ha='left',
    va='top',
    fontsize=8,
)
ax.legend(loc='lower left', fontsize=8, frameon=True)

ax_img = fig.add_subplot(grid[0, 1])
try:
    from PIL import Image
    if PAPER_PANEL_PLOT.exists():
        img = Image.open(PAPER_PANEL_PLOT)
        wimg, himg = img.size
        crop = img.crop((int(0.765 * wimg), 0, wimg, himg))
        ax_img.imshow(crop)
        ax_img.set_title('Paper Fig. 4, 4x4 panel')
    else:
        ax_img.text(0.5, 0.5, 'Paper panel image not found', ha='center', va='center')
except Exception as exc:
    ax_img.text(0.5, 0.5, f'Could not load paper panel:\n{exc}', ha='center', va='center')
ax_img.axis('off')

fig.suptitle('DES Y3 tomo 4 shear auto spectrum: paper-style error-bar check', y=1.02, fontsize=13)
fig.savefig(paper_style_plot_path, dpi=190, bbox_inches='tight')
plt.close(fig)

paper_style_summary = {
    'plot': str(paper_style_plot_path.relative_to(PACKAGE_ROOT)),
    'covariance_label': paper_covariance_label,
    'covariance_space': paper_cov['covariance_space'],
    'first_five_ell_eff': ell_eff[:5].tolist(),
    'first_five_ell_cl_1e7': y_ee[:5].tolist(),
    'first_five_ell_yerr_1e7': yerr_ee[:5].tolist(),
}
summary['paper_style_corrected_covariance'] = paper_style_summary
summary.setdefault('plots', {})['tomo4_cls_paper_style_corrected_covariance'] = str(paper_style_plot_path.relative_to(PACKAGE_ROOT))
save_json(OUTDIR / 'summary.json', summary)

print(json.dumps(to_builtin(paper_style_summary), indent=2, sort_keys=True))
print('Wrote', paper_style_plot_path)

## 8. Interpretation Checklist

After running the notebook, inspect `summary.json` first.

Interpretation logic:

- If `map_residuals_tomo4` has large residuals, the transfer HDF5 map is not faithful to the processed pickle.
- If raw and normalized weighted-mask spectra disagree, the mask/noise normalization is inconsistent.
- If the random-rotation mean disagrees with the analytic noise, the shape-noise scalar or subtraction convention is wrong.
- If only `current_suspect_pseudo_div_mean_mask2_uncoupled` has much larger errors, the broad Fig. 4 error bars are caused by the covariance input convention.
- If all covariance variants are broad, compare the mode-count estimate, `fsky_eff`, BB power, and the exact binning/pixel-window choices.

In [13]:
# Final machine-readable summary and a compact text report.
summary['finished_unix_time'] = time.time()
summary['diagnostic_outputs'] = {
    'summary_json': str((OUTDIR / 'summary.json').relative_to(PACKAGE_ROOT)),
    'map_residuals_tomo4_json': str((OUTDIR / 'map_residuals_tomo4.json').relative_to(PACKAGE_ROOT)),
    'tomo4_shear_maps_quicklook_png': str((OUTDIR / 'tomo4_shear_maps_quicklook.png').relative_to(PACKAGE_ROOT)),
    'tomo4_cls_covariance_variants_png': str((OUTDIR / 'tomo4_cls_covariance_variants.png').relative_to(PACKAGE_ROOT)),
    'tomo4_cls_paper_style_corrected_covariance_png': str((OUTDIR / 'tomo4_cls_paper_style_corrected_covariance.png').relative_to(PACKAGE_ROOT)),
    'tomo4_covariance_correlation_matrices_png': str((OUTDIR / 'tomo4_covariance_correlation_matrices.png').relative_to(PACKAGE_ROOT)),
    'tomo4_noise_rotation_check_png': str((OUTDIR / 'tomo4_noise_rotation_check.png').relative_to(PACKAGE_ROOT)),
}

# Add a simple conclusion flag that should be refined after inspecting the plots.
if summary.get('shape_noise_check') is None and 'noise_check' in globals():
    summary['shape_noise_check'] = noise_check

conclusion = 'incomplete'
if summary.get('map_residuals_tomo4', {}).get('ran'):
    count_ok = summary['map_residuals_tomo4']['count']['max_abs'] == 0.0
    g1_ok = summary['map_residuals_tomo4']['gamma1_observed_union']['p99_abs'] < 1e-7
    g2_ok = summary['map_residuals_tomo4']['gamma2_namaster_observed_union']['p99_abs'] < 1e-7
    if count_ok and g1_ok and g2_ok:
        conclusion = 'map_matches_processed_pickle'
if summary.get('covariance_diagnostics', {}).get('ran'):
    ratio = summary['covariance_diagnostics'].get('suspect_over_corrected_yerr_ratio_median')
    if ratio is not None and ratio > 2.0:
        conclusion = 'covariance_input_convention_likely_problem'
    elif ratio is not None:
        coupled_spaces = [
            v.get('covariance_space') for v in summary['covariance_diagnostics'].get('variants', {}).values()
        ]
        if 'full_ell_coupled' in coupled_spaces:
            conclusion = 'map_noise_ok_full_ell_coupled_covariance_not_comparable_to_decoupled_bandpowers'
summary['provisional_conclusion'] = conclusion

save_json(OUTDIR / 'summary.json', summary)
print(json.dumps(to_builtin(summary['diagnostic_outputs']), indent=2, sort_keys=True))
print('Provisional conclusion:', conclusion)
print('Summary:', OUTDIR / 'summary.json')

{
  "map_residuals_tomo4_json": "diagnostics/des_y3_shear_tomo4_covariance/map_residuals_tomo4.json",
  "summary_json": "diagnostics/des_y3_shear_tomo4_covariance/summary.json",
  "tomo4_cls_covariance_variants_png": "diagnostics/des_y3_shear_tomo4_covariance/tomo4_cls_covariance_variants.png",
  "tomo4_covariance_correlation_matrices_png": "diagnostics/des_y3_shear_tomo4_covariance/tomo4_covariance_correlation_matrices.png",
  "tomo4_noise_rotation_check_png": "diagnostics/des_y3_shear_tomo4_covariance/tomo4_noise_rotation_check.png",
  "tomo4_shear_maps_quicklook_png": "diagnostics/des_y3_shear_tomo4_covariance/tomo4_shear_maps_quicklook.png"
}
Provisional conclusion: map_matches_processed_pickle
Summary: /global/cfs/cdirs/lsst/www/shivamp/DESI/act_desi_ksz_transfer/diagnostics/des_y3_shear_tomo4_covariance/summary.json
